In [1]:
#! nvidia-smi

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import itertools
import time
# import matplotlib.pyplot as plt

In [3]:
class MIF(nn.Module):

    def __init__(
        self,
        R_on = 1000,
        R_off = 100000,
        v_on = 110,
        v_off = 5,
        tau = 100,
        E1 = 0,
        E2 = 50,
        C = 100 * 10**(-6),
        k_th = 0.6 * 25,
    ):
        super(MIF, self).__init__()

        self.R_on = R_on
        self.R_off = R_off
        self.v_on = v_on
        self.v_off = v_off
        self.tau = tau
        self.E1 = E1
        self.E2 = E2
        self.C = C
        self.k_th = k_th


    def forward(self, _input, x1, x2, G1, G2, v):
        v = (_input-G1*(v-self.E1)-G2*(v-self.E2))/self.C + v
        x1 = 1/self.tau*( (1-x1)/(1+torch.exp((self.v_on-(v-self.E1))/self.k_th)) - x1/(1+torch.exp(((v-self.E1)-self.v_off)/self.k_th))  ) + x1 #v[t] or v[t+1] both fine
        x2 = 1/self.tau*( (1-x2)/(1+torch.exp((self.v_on-(v-self.E2))/self.k_th)) - x2/(1+torch.exp(((v-self.E2)-self.v_off)/self.k_th))  ) + x2 #v[t] or v[t+1] both fine
        G1 = x1/self.R_on + (1-x1)/self.R_off
        G2 = x2/self.R_on + (1-x2)/self.R_off

        return x1, x2, G1, G2, v


    def init_MIF(self, batch_size, *args):
        x1 = torch.ones((batch_size, *args), device=device, dtype=dtype) * 0.0238
        x2 = torch.ones((batch_size, *args), device=device, dtype=dtype) * 0.0238
        G1 = x1 / self.R_on + (1-x1)/self.R_off
        G2 = x2 / self.R_on + (1-x2)/self.R_off
        # v = torch.ones((batch_size, *args), device=device, dtype=dtype) * (self.E1 + self.E2)/2
        v = torch.zeros((batch_size, *args), device=device, dtype=dtype)

        return x1, x2, G1, G2, v

In [4]:
# Training Parameters
batch_size = 128
data_path='./data'

dtype = torch.float
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [5]:
# Define a transform
transform = transforms.Compose([
            transforms.Resize((28, 28)),
            transforms.Grayscale(),
            transforms.ToTensor(),
            transforms.Normalize((0,), (1,))])

In [6]:
mnist_train = datasets.MNIST(data_path, train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(data_path, train=False, download=True, transform=transform)

# Create DataLoaders
train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=True, drop_last=True)

In [7]:
# Network Architecture
num_inputs = 28*28
num_hidden = 100
num_output = 10

scale1 = 128
scale2 = 4096
alpha = 128
num_steps = 1000
dis_steps = 100
input_steps = range(0, num_steps, dis_steps)

In [8]:
class Alpha(nn.Module):

    def __init__(
        self,
        tau_alpha = alpha,   
    ):
        super(Alpha, self).__init__()

        self.tau_alpha = tau_alpha

    def forward(self, _input, a, I):
        a = -a/self.tau_alpha + _input
        I = (a-I)/self.tau_alpha + I
        return a, I

    def init_Alpha(self, batch_size, *args):
        I = torch.zeros((batch_size, *args), device=device, dtype=dtype)
        a = torch.zeros((batch_size, *args), device=device, dtype=dtype)
        return a, I,

In [9]:
# from google.colab import drive
# drive.mount('/content/drive')

In [10]:
# Define Network
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        # Initialize layers
        self.alpha0 = Alpha()
        self.fc1 = nn.Linear(num_inputs, num_hidden)
        self.mif1 = MIF()
        self.fc2 = nn.Linear(num_hidden, num_output)
        self.mif2 = MIF()

    def forward(self, x):

        # Initialize hidden states and outputs at t=0
        a0, I0 = self.alpha0.init_Alpha(batch_size, num_inputs)
        x1_h, x2_h, G1_h, G2_h, v_h = self.mif1.init_MIF(batch_size, num_hidden)
        x1_out, x2_out, G1_out, G2_out, v_out = self.mif2.init_MIF(batch_size, num_output)  

        # Record the final layer
        v_o_rec = []

        inp0 = torch.zeros((1,num_inputs)).to(device)
        
        for step in range(num_steps):

            if step in input_steps:
              a0, I0 = self.alpha0(x, a0, I0)
            else:
              a0, I0 = self.alpha0(inp0, a0, I0)

            out = self.fc1(I0)
            out = out/scale1

            x1_h, x2_h, G1_h, G2_h, v_h = self.mif1(out, x1_h, x2_h, G1_h, G2_h, v_h)

            out = self.fc2(v_h)
            out = out/scale2

            x1_out, x2_out, G1_out, G2_out, v_out = self.mif2(out, x1_out, x2_out, G1_out, G2_out, v_out)
                
            v_o_rec.append(v_out)

        return torch.stack(v_o_rec, dim=0)

In [11]:
from torch.nn import CrossEntropyLoss
from torch.optim import Adam

net = Net().to(device)

optimizer = torch.optim.Adam(net.parameters(), lr=1e-4, betas=(0.9, 0.999))
log_softmax_fn = nn.LogSoftmax(dim=-1)
loss_fn = nn.NLLLoss()

In [12]:
num_epochs = 100

test_acc_hist = []

for epoch in range(num_epochs):
    # train
    start_time = time.time()
    train_batch = iter(train_loader)

    for data_it, targets_it in train_batch:
        data_it = data_it.to(device)
        targets_it = targets_it.to(device)

        v_rec = net(data_it.view(batch_size, -1)) # Time x Batch x num_output_neuron
        log_p_y = log_softmax_fn(v_rec)

        loss_val = torch.zeros((1), dtype=dtype, device=device)
        for step in range(num_steps):
            loss_val += loss_fn(log_p_y[step], targets_it)

        optimizer.zero_grad()
        loss_val.backward()
        optimizer.step()

    epoch_time = time.time() - start_time

    # test
    correct = 0
    total = 0

    with torch.no_grad():
        net.eval()

        for test_data, test_labels in test_loader:
            test_data, test_labels = test_data.to(device), test_labels.to(device)
            test_v = net(test_data.view(batch_size, -1))
            t_max, _ = test_v.max(dim=0) # Batch x num_output_neuron # find max Vmem for each MIF neuron through time
            _, idx = t_max.max(dim=-1) # Batch # find the index of the MIF neuron with max Vmem
            acc = np.mean((test_labels == idx).detach().cpu().numpy())

            total += test_labels.size(0)
            correct += acc * test_labels.size(0)
    
        test_acc = 100 * correct / total
        test_acc_hist.append(test_acc)

    test_time = time.time() - start_time

    print(f'Epoch: {epoch} \t Test Accuracy: {test_acc} \t Train time: {epoch_time} \t Total time: {test_time}')

    # np.savetxt(f'/content/drive/MyDrive/multi_mif/2layer_noalpha/3.2/test_acc_{epoch}.txt', test_acc_hist)


KeyboardInterrupt: 